In [2]:
!pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   - -------------------------------------- 4.5/100.2 MB 24.1 MB/s eta 0:00:04
   --- ------------------------------------ 7.9/100.2 MB 20.5 MB/s eta 0:00:05
   ---- ----------------------------------- 11.3/100.2 MB 18.9 MB/s eta 0:00:05
   ----- ---------------------------------- 14.7/100.2 MB 18.0 MB/s eta 0:00:05
   ------- -------------------------------- 18.4/100.2 MB 17.7 MB/s eta 0:00:05
   ------- -------------------------------- 18.9/100.2 MB 17.7 MB/s eta 0:00:05
   --------- ------------------------------ 23.1/100.2 MB 15.8 MB/s eta 0:00:05
   ---------- ----------------------------- 27.3/100.2 MB 16.3 MB/s eta 0:00:05
   ------------ --------------------------- 30.7/100.2 MB 16.3 MB/s eta 0:00:05
   ------------- -------------------------- 34.1/100.2 MB 16.3 MB/s eta 0:00:05
   -------------- ------------------------- 37.5/100.2 MB 16.3 MB/s 

In [3]:
#importing the necessary packages 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
%matplotlib inline
import warnings 

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error , r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor , AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression , Ridge ,Lasso
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [8]:
df=pd.read_csv('../data/cleaned_earthquake.csv')

In [9]:
df

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,depthError,magNst,status,locationSource,magSource,day,gap_conservative,dmin_conservative,gap_median,dmin_median
0,2024-01-26 04:52:42.967000+00:00,31.604000,-104.213000,4.4198,1.70,ml,18.0,69.00,0.100000,0.50,...,1.292059,13.0,automatic,tx,tx,2024-01-26,69.0,0.100000,69.00,0.100000
1,2024-01-26 04:42:50.711000+00:00,64.501000,-146.905800,4.2000,1.40,ml,1.0,89.91,0.059530,0.75,...,0.200000,1.0,automatic,ak,ak,2024-01-26,360.0,5.000000,89.91,0.059530
2,2024-01-26 04:32:51.471000+00:00,63.529000,-147.554300,13.1000,1.60,ml,1.0,89.91,0.059530,0.62,...,0.300000,1.0,automatic,ak,ak,2024-01-26,360.0,5.000000,89.91,0.059530
3,2024-01-26 04:29:01.180000+00:00,38.833168,-122.797165,1.7300,0.40,md,9.0,65.00,0.007468,0.02,...,0.970000,10.0,automatic,nc,nc,2024-01-26,65.0,0.007468,65.00,0.007468
4,2024-01-26 04:23:14.444000+00:00,63.546200,-150.971900,0.0000,1.20,ml,1.0,89.91,0.059530,0.80,...,0.400000,1.0,automatic,ak,ak,2024-01-26,360.0,5.000000,89.91,0.059530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8781,2023-12-27 05:39:05.490000+00:00,18.097333,-66.835500,18.3100,2.14,md,6.0,106.00,0.128500,0.08,...,1.420000,6.0,reviewed,pr,pr,2023-12-27,106.0,0.128500,106.00,0.128500
8782,2023-12-27 05:37:35.928000+00:00,54.771700,-164.123600,9.6690,3.60,mb,47.0,91.00,0.002000,0.70,...,4.711000,5.0,reviewed,us,us,2023-12-27,91.0,0.002000,91.00,0.002000
8783,2023-12-27 05:34:36.680000+00:00,54.748000,-164.104667,4.2000,0.67,ml,4.0,211.00,0.059530,0.13,...,1.840000,4.0,reviewed,av,av,2023-12-27,211.0,5.000000,211.00,0.059530
8784,2023-12-27 05:32:44.484000+00:00,43.462200,16.286000,10.0000,2.70,ml,17.0,78.00,1.440000,0.56,...,1.990000,44.0,reviewed,us,us,2023-12-27,78.0,1.440000,78.00,1.440000


In [10]:
df.isnull().sum()

time                 0
latitude             0
longitude            0
depth                0
mag                  0
magType              0
nst                  0
gap                  0
dmin                 0
rms                  0
net                  0
id                   0
updated              0
place                0
type                 0
depthError           0
magNst               0
status               0
locationSource       0
magSource            0
day                  0
gap_conservative     0
dmin_conservative    0
gap_median           0
dmin_median          0
dtype: int64

preparing the X and y variable for the regression only models 


In [12]:
X=df.drop(columns=['mag'],axis=1)
X.head(2)

,time,latitude,longitude,depth,magType,nst,gap,dmin,rms,net,...,depthError,magNst,status,locationSource,magSource,day,gap_conservative,dmin_conservative,gap_median,dmin_median
0,2024-01-26 04:52:42.967000+00:00,31.604,-104.2130,4.4198,ml,18.0,69.00,0.10000,0.50,tx,...,1.292059,13.0,automatic,tx,tx,2024-01-26,69.0,0.1,69.00,0.10000
1,2024-01-26 04:42:50.711000+00:00,64.501,-146.9058,4.2000,ml,1.0,89.91,0.05953,0.75,ak,...,0.200000,1.0,automatic,ak,ak,2024-01-26,360.0,5.0,89.91,0.05953


In [13]:
y=df['mag']

In [14]:
y

0       1.70
1       1.40
2       1.60
3       0.40
4       1.20
        ... 
8781    2.14
8782    3.60
8783    0.67
8784    2.70
8785    3.00
Name: mag, Length: 8786, dtype: float64

In [15]:
#Create column transformer with 3 types of transformers 
num_features=X.select_dtypes(exclude='object').columns
cat_features=X.select_dtypes(include='object').columns

from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder()

preprocessor=ColumnTransformer(
    [
        ('OneHotEncoder',oh_transformer,cat_features),
        ('StandardScaler', numeric_transformer,num_features),
    ]
)

In [16]:
X=preprocessor.fit_transform(X)

In [17]:
X.shape

(8786, 31327)

Separting the dataset into train test split 

In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)
X_train.shape , X_test.shape

((6589, 31327), (2197, 31327))

Creating an evaluating function to give all the metrics after the model training 


In [20]:
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2_square=r2_score(true,predicted)
    return mae,rmse,r2_square

In [23]:
models={
    "Linear Regression":LinearRegression(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "K-Neighbours Regressor":KNeighborsRegressor(),
    "DecisionTree":DecisionTreeRegressor(),
    "Random Forest Regressor":RandomForestRegressor(),
    "XGBRegressor":XGBRegressor(),
    "CatBoosting Regressor":CatBoostRegressor(verbose=False),
    "AdaBoost Regressor":AdaBoostRegressor()
}

model_list=[]
r2_list=[]

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

    #making the predictions
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    #evaluating 
    model_train_mae,model_train_rmse,model_train_r2=evaluate_model(y_train,y_train_pred)
    model_test_mae,model_test_rmse,model_test_r2=evaluate_model(y_test,y_test_pred)


    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 0.0002
- Mean Absolute Error: 0.0001
- R2 Score: 1.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.4500
- Mean Absolute Error: 0.3374
- R2 Score: 0.8709


Lasso
Model performance for Training set
- Root Mean Squared Error: 1.1680
- Mean Absolute Error: 0.8516
- R2 Score: 0.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 1.2542
- Mean Absolute Error: 0.9153
- R2 Score: -0.0030


Ridge
Model performance for Training set
- Root Mean Squared Error: 0.0939
- Mean Absolute Error: 0.0698
- R2 Score: 0.9935
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.4515
- Mean Absolute Error: 0.3387
- R2 Score: 0.8700


K-Neighbours Regressor
Model performance for Training set
- Root Mean Squared Error: 0.3469
- Mean Absolute Error: 0.2455
- R2 Score: 0.9118
---------------------

In [22]:
pd.DataFrame(list(zip(model_list,r2_list)),columns=['Model Name','R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
5,Random Forest Regressor,0.922147
6,XGBRegressor,0.921036
7,CatBoosting Regressor,0.917720
3,K-Neighbours Regressor,0.876774
4,DecisionTree,0.871195
0,Linear Regression,0.870884
2,Ridge,0.870043
8,AdaBoost Regressor,0.787004
1,Lasso,-0.002971
